In [19]:
from getpass import getuser
import subprocess
subprocess.run(['pip', 'install', 'sparky_bc'], check=True)
from sparky_bc import Sparky
from helper.helper import Helper
#from vspc_config_utils import Config_utils 
#Config_utils para subir tablas, instalarla en la terminal en caso de usarse: pip install vspc_config_utils
import pandas as pd
 
sp = Sparky(username = getuser(),dsn="impala-prod")
ih = Helper(username=getuser(),dsn='impala-prod')
#c_util = Config_utils(sparky=sp)



In [20]:
# Leer la consulta del archivo SQL
from pathlib import Path

sql_path = Path("sql/lz/consulta_data_liq_proc.sql")
with open(sql_path, 'r') as f:
    query = f.read().strip()

print(f"Consulta a ejecutar:\n{query}\n")

# Cargar el resultado en un DataFrame usando el Helper
ih.fetch_size = 10000
try:
    df = ih.obtener_dataframe(query)
except Exception as e:
    # Detect common ODBC IM002 DSN-missing error and provide a fallback (empty DataFrame)
    msg = str(e)
    if "IM002" in msg or "No se encuentra el nombre del origen de datos" in msg or "Atributo de cadena de conexión no válido" in msg:
        print("Advertencia: DSN 'impala-prod' no encontrado o cadena de conexión inválida. Se crea un DataFrame vacío como fallback.")
        # Columnas esperadas por la consulta
        df = pd.DataFrame(columns=['id_epica', 'periodo', 'codigo_proceso', 'procedimiento', 'actividad'])
    else:
        raise
print(f"DataFrame cargado con {len(df)} filas y {len(df.columns)} columnas")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()
df.info()

2026-06-18 11:55:42,470 INFO Ejecutando consulta a través de Helper


Consulta a ejecutar:
SELECT
    t1.id_epica,
    t1.periodo,
    t2.cod_proc AS codigo_proceso,
    t2.procedimiento,
    t2.actividad
FROM (
SELECT DISTINCT 
        ig.id_epica, 
        pb.codigo_proceso, 
        ig.periodo
    FROM resultados_vspc_dise.cdeproc_sabana_procesos_beneficios pb
    JOIN resultados_vspc_dise.cdeproc_sabana_informacion_general ig
        ON ig.id_epica = pb.id_epica
    WHERE ig.periodo = '2026Q1'
) t1
JOIN (
    SELECT 
        cod_proc,
        procedimiento,
        actividad
    FROM resultados_serv_para_los_clientes.bpms_data_liquida
    WHERE cod_proc IS NOT NULL
      AND tipo_actividad IN ('TAREA', 'CONTROL', 'CONTROL PARCIAL', 'PROCEDIMIENTO')
      AND cod_proc NOT LIKE '%CM%'
    GROUP BY 
        cod_proc,
        procedimiento,
        actividad
) t2
ON t1.codigo_proceso = t2.cod_proc

Advertencia: DSN 'impala-prod' no encontrado o cadena de conexión inválida. Se crea un DataFrame vacío como fallback.
DataFrame cargado con 0 filas y 5 column

In [16]:
# Cargar arquetipos y hacer matching con los resultados SQL
import sys, os
# Asegura que la carpeta del notebook esté en sys.path
sys.path.insert(0, os.getcwd())

# Intentar import normal, con fallback a carga por ruta si falla
try:
    from helper.match_arquetipos import load_arquetipos, match_codes
except ModuleNotFoundError:
    import importlib.util
    mod_path = os.path.join(os.getcwd(), 'helper', 'match_arquetipos.py')
    spec = importlib.util.spec_from_file_location('match_arquetipos', mod_path)
    match_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(match_mod)
    load_arquetipos = match_mod.load_arquetipos
    match_codes = match_mod.match_codes

# Ajusta la ruta si es necesario
df_arq, df_arq_expl = load_arquetipos('data/arquetipos.csv')

# `df` debe existir en el notebook (resultado de la consulta SQL)
result, merged = match_codes(df, df_arq_expl, sql_code_col='codigo_proceso')
print(f"Filas SQL: {len(df)}; coincidencias en merged (filas con proc_code): {merged['proc_code'].notna().sum()}")
# Filtrar solo filas con coincidencias en arquetipos
result_filtered = result[result['matched_codigo_arquetipo'].notna()]
result_filtered

Filas SQL: 0; coincidencias en merged (filas con proc_code): 0


,id_epica,periodo,codigo_proceso,procedimiento,actividad,matched_proc_codes,matched_codigo_arquetipo,matched_nombre_arquetipo


In [17]:
# Guardar resultados en CSV
import os
from datetime import datetime

# Check if result_filtered exists and has data
if 'result_filtered' not in locals() or result_filtered is None or len(result_filtered) == 0:
	print("Advertencia: No hay resultados con coincidencias para guardar.")
else:
	output_dir = 'data/output'
	os.makedirs(output_dir, exist_ok=True)
	# Usar timestamp para evitar conflictos
	timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
	output_path = os.path.join(output_dir, f'resultado_matching_arquetipos_{timestamp}.csv')
	result_filtered.to_csv(output_path, index=False, encoding='utf-8')
	print(f"Archivo guardado en: {output_path}")
	print(f"Total de filas: {len(result_filtered)}")

Advertencia: No hay resultados con coincidencias para guardar.
